# Phase 2 — **Scheduled QAT (Cosine)** — Google Colab / Kaggle TPU

**What is Scheduled QAT?** Standard QAT applies full quantization noise from step 1. Our hypothesis: gradually reducing the simulated bit-width over training (FP32 → 16 → 8 → 4) gives weights time to adapt at each level. This notebook tests the **cosine schedule** — slow start, fast middle, slow end (borrowed from cosine LR decay).

**Three schedules** available in separate notebooks:
- `04_scheduled_qat_linear.ipynb` — constant slope
- `04_scheduled_qat_cosine.ipynb` (this one) — slow→fast→slow
- `04_scheduled_qat_step.ipynb` — hard drops at fixed epochs

**Inputs:** FP32 baseline logits from notebook 01.

**Outputs (saved locally and to Drive):**
- `models/checkpoints/scheduled_qat_cosine_int4.pt` — final trained model
- `results/scheduled_qat_cosine_int4/training_results.json` — metrics
- `results/logs/scheduled_cosine_int4/per_step_loss.jsonl` — training dynamics
- `results/scheduled_qat_cosine_int4_inference.json` — sample generations

**Estimated time:**
- **Colab T4:** ~1.5 hours (BF16 emulation + memory tricks)
- **Kaggle TPU v3-8:** ~20 minutes (native BF16 support)

## 1. Setup

In [ ]:
import os, sys, subprocess, json, gc
from pathlib import Path

# Detect runtime and mount Drive only on Colab
IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle')
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKING_DIR = "/content"
    DRIVE_ROOT = "/content/drive/MyDrive/sqat-scheduled-qat"
    BASELINE_DIR = "/content/drive/MyDrive/sqat-baseline/results/baseline"
elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"
    DRIVE_ROOT = "/kaggle/working/sqat-scheduled-qat"
    BASELINE_DIR = "/kaggle/input/sqat-baseline/results/baseline"  # from dataset
else:
    WORKING_DIR = "."
    DRIVE_ROOT = "./sqat-scheduled-qat-output"
    BASELINE_DIR = "./results/baseline"

REPO_DIR = f"{WORKING_DIR}/scheduled-qat-for-slm"
GITHUB_URL = "https://github.com/JpCurada/scheduled-qat-for-slm.git"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {GITHUB_URL} to {REPO_DIR}...")
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

FP32_LOGITS = Path(BASELINE_DIR) / "fp32_logits.pt"
if not FP32_LOGITS.exists():
    print(f"⚠️  FP32 logits not found at {FP32_LOGITS}. Run notebook 01 first.")
else:
    print(f"✓ FP32 logits found: {FP32_LOGITS}")

print(f"\nRuntime: {'Colab' if IS_COLAB else 'Kaggle TPU' if IS_KAGGLE else 'Local'}")
print(f"Working dir: {WORKING_DIR}")
print(f"Repo: {REPO_DIR}")

In [ ]:
# Install dependencies: [viz] for plots, [mem] for 8-bit AdamW + bitsandbytes
!pip install -q -e ".[viz,mem]"
import torch

device_name = "TPU" if IS_KAGGLE else (torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU")
print(f"Device: {device_name}")

## 2. Config & Memory-Efficient Training

In [ ]:
import yaml
from src.quantization.scheduler import build_scheduler

# Determine model cache and epochs
if IS_COLAB:
    # Try Drive first, then local
    DRIVE_MODEL_CACHE = "/content/drive/MyDrive/sqat-baseline/models/base"
    LOCAL_MODEL_CACHE = f"{REPO_DIR}/models/base"
    MODEL_CACHE = DRIVE_MODEL_CACHE if Path(DRIVE_MODEL_CACHE).exists() else LOCAL_MODEL_CACHE
    KAGGLE_EPOCHS = 1  # Compressed for faster Colab demo
    ORIGINAL_EPOCHS = 3
elif IS_KAGGLE:
    MODEL_CACHE = f"{WORKING_DIR}/models/base"  # Local disk in Kaggle
    KAGGLE_EPOCHS = 1  # Compressed for TPU memory constraints (OOM at 3 epochs)
    ORIGINAL_EPOCHS = 3
else:
    MODEL_CACHE = f"{REPO_DIR}/models/base"
    KAGGLE_EPOCHS = 1
    ORIGINAL_EPOCHS = 3

SCHEDULE_SCALE = KAGGLE_EPOCHS / ORIGINAL_EPOCHS

# Memory-efficient training (BF16 + 8-bit AdamW on Colab, standard AdamW on TPU)
# Gradient checkpointing disabled on XLA (Kaggle TPU) due to compatibility
MEM_EFFICIENT = True

# Config patch function
def patch_config_cosine():
    with open("configs/scheduled_qat/scheduled_cosine_int4.yaml") as f:
        cfg = yaml.safe_load(f)
    
    cfg["model"]["cache_dir"] = MODEL_CACHE
    cfg["data"]["seq_length"] = 128  # Optimized: 16× reduction from 2048, further reduced from 256 for TPU XLA memory
    cfg["training"]["epochs"] = KAGGLE_EPOCHS
    
    if IS_COLAB:
        cfg["training"]["batch_size"] = 2      # Colab T4: 14GB
        cfg["training"]["gradient_accumulation_steps"] = 8
    else:  # Kaggle TPU or local
        cfg["training"]["batch_size"] = 2      # TPU: reduce to 2 to avoid XLA OOM (was 4)
        cfg["training"]["gradient_accumulation_steps"] = 8  # increase accum to maintain effective batch
    
    cfg["training"]["warmup_steps"] = 50  # Reduced from 100
    
    if MEM_EFFICIENT:
        cfg["training"]["compute_dtype"] = "bf16"      # BF16 weight training
        cfg["training"]["use_amp"] = False             # no GradScaler with BF16 weights
        cfg["training"]["use_8bit_optimizer"] = not IS_KAGGLE  # 8-bit optimizer not compatible with XLA; use standard AdamW on TPU
        cfg["training"]["gradient_checkpointing"] = False  # Disable on XLA/TPU
    
    cfg["logging"]["save_every_steps"] = 999999  # only save final checkpoint
    cfg["logging"]["eval_every_steps"] = 250  # More frequent checks
    cfg["logging"]["log_dir"] = f"{REPO_DIR}/results/logs/scheduled_cosine_int4/"
    cfg["export"]["output_dir"] = f"{REPO_DIR}/models/gguf/"
    
    # Scale schedule transitions from 3 epochs to KAGGLE_EPOCHS
    sch = cfg["schedule"]
    sch["warmup_epochs"] = round(sch["warmup_epochs"] * SCHEDULE_SCALE, 4)
    sch["stabilization_epochs"] = round(sch["stabilization_epochs"] * SCHEDULE_SCALE, 4)
    for t in sch.get("transitions") or []:
        t["epoch"] = round(t["epoch"] * SCHEDULE_SCALE, 4)
    
    return cfg

cfg_cosine = patch_config_cosine()
print(f"Config: {KAGGLE_EPOCHS} epochs | {cfg_cosine['training']['batch_size']} batch | accum={cfg_cosine['training']['gradient_accumulation_steps']} | BF16={MEM_EFFICIENT}")
print(f"Effective batch: {cfg_cosine['training']['batch_size'] * cfg_cosine['training']['gradient_accumulation_steps']}")
print(f"Model cache: {MODEL_CACHE}")
print(f"8-bit optimizer: {cfg_cosine['training']['use_8bit_optimizer']} (disabled on TPU due to XLA incompatibility)")
print(f"Schedule transitions (cosine):")
for t in cfg_cosine["schedule"].get("transitions", []):
    print(f"  epoch {t['epoch']:.2f} → INT{t['bits']}")

## 3. Visualize the Cosine Schedule

In [ ]:
import numpy as np
import plotly.graph_objects as go
from src.quantization.scheduler import snap_bits, ScheduleConfig

# Build scheduler from the patched config
schedule_dict = cfg_cosine["schedule"]
schedule_config = ScheduleConfig(**schedule_dict)

sched = build_scheduler(schedule_config, total_epochs=float(cfg_cosine["training"]["epochs"]))

# Sample schedule over training
epochs = np.linspace(0, cfg_cosine["training"]["epochs"], 400)
continuous = [sched.get_state(e).continuous_bits for e in epochs]
snapped = [snap_bits(c) or 32 for c in continuous]  # Actual fake-quant levels

fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs, y=continuous, name="continuous (schedule)",
                          line=dict(color="#DD8452", width=2)))
fig.add_trace(go.Scatter(x=epochs, y=snapped, name="fake-quant level (actual)",
                          line=dict(color="#C44E52", shape="hv", width=2)))
fig.update_xaxes(title_text="epoch")
fig.update_yaxes(title_text="bit-width", tickvals=[4, 8, 16, 32])
fig.update_layout(
    title="Cosine Schedule: Slow→Fast→Slow Precision Reduction (INT4 target)",
    height=450, hovermode="x unified",
    margin=dict(t=80, b=40, l=60, r=20)
)
fig.show()

print(f"Schedule: {schedule_config.type} | warmup={schedule_config.warmup_epochs:.2f}e | "
      f"target_bits={schedule_config.target_bits} | stab={schedule_config.stabilization_epochs:.2f}e")

## 4. Train with Cosine Schedule — INT4

In [ ]:
# Save config to a temp file so trainer can read it
import tempfile
import yaml

with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False, dir=REPO_DIR) as f:
    yaml.safe_dump(cfg_cosine, f, sort_keys=False)
    cfg_path = f.name

print(f"Config saved to: {cfg_path}")
print(f"\nStarting training...\n")

from src.training.trainer import run_experiment

results = run_experiment(
    config_path=cfg_path,
    device_str="cuda" if torch.cuda.is_available() else ("xla" if IS_KAGGLE else "cpu"),
    baseline_logits=str(FP32_LOGITS) if FP32_LOGITS.exists() else None,
    run_benchmarks=False,  # Skip benchmarks for speed
)

print("\n" + "="*70)
print("TRAINING COMPLETE (Cosine Schedule INT4)")
print("="*70)
for k, v in results.items():
    if not isinstance(v, (dict, list)):
        print(f"  {k:30s}  {v}")

# Clean up temp config
Path(cfg_path).unlink(missing_ok=True)
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 5. Sample Inference — Cosine Schedule vs FP32

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.quantization.scheduled_qat import load_checkpoint as load_sqat_ckpt

SAMPLE_PROMPTS = [
    "The capital of France is",
    "Python is a programming language",
    "Once upon a time, in a small village,",
    "Artificial intelligence can help",
]
MAX_NEW_TOKENS = 30
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-1.7B", cache_dir=MODEL_CACHE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def generate(model, prompts, max_tokens=MAX_NEW_TOKENS):
    model.eval()
    outputs = []
    for prompt in prompts:
        ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
        with torch.no_grad():
            gen = model.generate(ids, max_new_tokens=max_tokens, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
        outputs.append(tokenizer.decode(gen[0][ids.shape[1]:], skip_special_tokens=True).strip())
    return outputs

# Generate with FP32 baseline
print("Generating with FP32 baseline...")
fp32_model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-1.7B", cache_dir=MODEL_CACHE, torch_dtype=torch.float16
).to(device)
fp32_outs = generate(fp32_model)
del fp32_model; gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# Generate with trained Cosine Schedule INT4 model
print("Generating with Scheduled QAT (cosine) INT4...")
from src.models.model_wrapper import build_model_for_training
from src.utils.config_loader import load_config

# Reload config and build model
wrap = build_model_for_training(cfg_cosine, device, total_steps=1000)

# Load checkpoint
ckpt_path = Path(REPO_DIR) / "models" / "checkpoints" / "scheduled_qat_cosine_int4" / "final.pt"
if ckpt_path.exists():
    load_sqat_ckpt(wrap.model, str(ckpt_path), device)
    print(f"  Loaded checkpoint: {ckpt_path}")
else:
    print(f"  WARNING: Checkpoint not found: {ckpt_path}")

cosine_outs = generate(wrap.model)
del wrap; gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# Display side-by-side
import pandas as pd
inference_df = pd.DataFrame({
    "Prompt": SAMPLE_PROMPTS,
    "FP32 (baseline)": fp32_outs,
    "Cosine INT4": cosine_outs,
})

print("\n" + "="*70)
print(inference_df.to_string(index=False))
print("="*70)

## 6. Save Results

In [ ]:
# Save inference results
results["inference"] = {
    "prompts": SAMPLE_PROMPTS,
    "fp32": fp32_outs,
    "cosine_int4": cosine_outs,
}

result_dir = Path(REPO_DIR) / "results" / "scheduled_qat_cosine_int4"
result_dir.mkdir(parents=True, exist_ok=True)

result_file = result_dir / "training_results.json"
with result_file.open("w") as f:
    json.dump(results, f, indent=2, default=str)
print(f"Results saved: {result_file}")

# List checkpoints
print(f"\nCheckpoints:")
ckpt_dir = Path(REPO_DIR) / "models" / "checkpoints"
for ckpt in ckpt_dir.rglob("*.pt"):
    size_mb = ckpt.stat().st_size / 1e6
    print(f"  {ckpt.relative_to(REPO_DIR)}  ({size_mb:.0f} MB)")

## Next: Run Other Schedules

Repeat this workflow with the remaining schedule:
- `04_scheduled_qat_step.ipynb` — hard drops at fixed epochs

Then compare all three in notebook `07_benchmarks.ipynb` using perplexity, KL divergence, and generation quality.